# Session 11: Object Detection

**CVI4IC - Summer Semester 2026**

Detecting and localizing objects in images with bounding boxes.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

## 1. IoU (Intersection over Union)

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# Example
box_gt = [50, 50, 200, 200]
box_pred = [80, 60, 220, 210]

iou = compute_iou(box_gt, box_pred)
print(f"IoU: {iou:.4f}")

# Visualize
canvas = np.ones((300, 300, 3), dtype=np.uint8) * 255
cv2.rectangle(canvas, (box_gt[0], box_gt[1]), (box_gt[2], box_gt[3]), (0, 255, 0), 2)
cv2.rectangle(canvas, (box_pred[0], box_pred[1]), (box_pred[2], box_pred[3]), (0, 0, 255), 2)
cv2.putText(canvas, f"IoU: {iou:.2f}", (10, 280), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Green=GT, Red=Prediction")
plt.axis("off")
plt.show()

## 2. Non-Maximum Suppression

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """Simple Non-Maximum Suppression."""
    order = np.argsort(scores)[::-1]
    keep = []
    
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        
        if len(order) == 1:
            break
        
        ious = np.array([compute_iou(boxes[i], boxes[j]) for j in order[1:]])
        remaining = np.where(ious < iou_threshold)[0]
        order = order[remaining + 1]
    
    return keep

# Example: multiple overlapping detections
boxes = np.array([
    [100, 100, 210, 210],
    [105, 98, 215, 208],
    [110, 105, 220, 215],
    [50, 50, 120, 120],
])
scores = np.array([0.9, 0.75, 0.8, 0.6])

keep = nms(boxes, scores, iou_threshold=0.3)
print(f"Kept boxes: {keep}")

# Visualize before and after NMS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax_idx, (title, indices) in enumerate([("Before NMS", range(len(boxes))), ("After NMS", keep)]):
    canvas = np.ones((300, 300, 3), dtype=np.uint8) * 240
    for i in indices:
        b = boxes[i]
        cv2.rectangle(canvas, (b[0], b[1]), (b[2], b[3]), (0, 0, 255), 2)
        cv2.putText(canvas, f"{scores[i]:.2f}", (b[0], b[1]-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 1)
    axes[ax_idx].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    axes[ax_idx].set_title(title)
    axes[ax_idx].axis("off")

plt.tight_layout()
plt.show()

## 3. YOLO with Ultralytics (Optional)

Install with: `pip install ultralytics`

In [ ]:
# Uncomment to run (requires ultralytics package)
# from ultralytics import YOLO
# 
# model = YOLO("yolo11n.pt")
# results = model("https://ultralytics.com/images/bus.jpg")
# 
# for r in results:
#     im = r.plot()
#     plt.figure(figsize=(10, 8))
#     plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
#     plt.axis("off")
#     plt.show()
print("Uncomment the code above after installing ultralytics")

## Exercises

1. Implement IoU for rotated bounding boxes (oriented bounding boxes)
2. Compute mAP for a set of predictions vs. ground truth annotations
3. Run YOLO on your own images and analyze the detection confidence scores

In [ ]:
# Your code here
